# Functions

This notebook contains a list of custom made functions to be used in other notebooks.

In [1]:
from matplotlib.colors import ListedColormap, Normalize

In [2]:
def get_cmap_bdrc():
    """
    Returns a colourmap scheme matching the BDRC colour scheme.
    """ 
    colors = [
        'white',
        (161/255, 237/255, 227/255),
        (92/255, 227/255, 186/255),
        (252/255, 215/255, 117/255),
        (218/255, 114/255, 48/255),
        (158/255, 98/255, 38/255),
        (113/255, 73/255, 33/255),
        (57/255, 37/255, 17/255),
        (29/255, 19/255, 9/255)
            ]
    cmap = ListedColormap(colors)
    return cmap

In [3]:
def get_custom_norm(levels):
    """
    Return a BoundaryNorm object for custom colorbar levels.

    Parameters:
    - levels: list of numeric boundaries (e.g., [0, 5, 20, 50, ...])

    Returns:
    - norm: BoundaryNorm object
    """
    return BoundaryNorm(boundaries=levels, ncolors=bdrc_cmap.N, extend='max')


In [1]:
def get_normalised_bdrc(dustvar, cmap):
    """
    Return a BoundaryNorm object for pre-determined BDRC colorbar levels .

    Parameters:
    - dustvar: Type of dust variable. Can be either dust optical depth or dust concentration.
    - cmap: BDRC colourmap, as returned by the cmap_bdrc() function.

    Returns:
    - norm: BoundaryNorm object
    """
    if dustvar == "od550_dust":
        levels = [0, 0.1, 0.2, 0.4, 0.8, 1.2, 1.6, 3.2, 6.4]
    elif dustvar == "sconc_dust":
        levels = [0, 5, 20, 50, 200, 500, 2000, 5000, 20000]
    elif dustvar == "dust_load":
        levels = [0, 0.1, 0.4, 0.8, 1.2, 1.6, 3.2, 6.4]
    elif dustvar == "ec550du":
        levels = [0, 5, 10, 25, 100, 250, 1000, 2500, 10000]
    else:
        raise ValueError(f"Unknown dust variable type: {var}. Please select either od550_dust, sconc_dust, dust_load or add a dust variable to the normalising function.")
    
    norm = BoundaryNorm(boundaries=levels, ncolors=cmap.N, extend='max')
    return norm


In [ ]:
def get_cmap_diff():
    """
    Returns a colourmap scheme matching MAPIES differenc plots.
    """ 
    cmap = plt.cm.coolwarm
    return cmap

In [1]:
def get_normalised_diff(dustvar, cmap):
    """
    Return a BoundaryNorm object for pre-determined diff colorbar levels .

    Parameters:
    - dustvar: Type of dust plot. Can be difference plot.
    - cmap: diff colourmap, as returned by the cmap_diff() function.

    Returns:
    - norm: BoundaryNorm object
    """

    if dustvar == "diff":
        #levels = [-0.9, -0.8, -0.7, -0.6, -0.5, -0.4, -0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
        levels = [-0.5, -0.4, -0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3, 0.4, 0.5]
    else:
        raise ValueError(f"Unknown dust variable type: {var}. Please select either od550_dust, sconc_dust, dust_load or add a dust variable to the normalising function.")
    
    norm = BoundaryNorm(boundaries=levels, ncolors=cmap.N, extend='both')
    return norm


In [ ]:
def get_cmap_num_obs():
    """
    Returns a colourmap scheme matching MAPIES num_obs plots.
    """ 
    cmap = plt.cm.rainbow
    return cmap

In [ ]:
def get_normalised_num_obs(dustvar, cmap):
    """
    Return a BoundaryNorm object for pre-determined diff colorbar levels .

    Parameters:
    - dustvar: Type of observational plot.
    - cmap: diff colourmap, as returned by the cmap_diff() function.

    Returns:
    - norm: BoundaryNorm object
    """

    if dustvar == "num_obs":
        #levels = [-0.9, -0.8, -0.7, -0.6, -0.5, -0.4, -0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
        levels = [0, 5, 10, 15, 20, 30, 40, 50]
    else:
        raise ValueError(f"Unknown dust variable type: {var}. Please select either od550_dust, sconc_dust, dust_load or add a dust variable to the normalising function.")
    
    norm = BoundaryNorm(boundaries=levels, ncolors=cmap.N, extend='max')
    return norm


In [5]:
def select_timesteps(ds):
    """
    Returns a slice of the dataset between index 4 and 12.
    """
    #return ds.isel(time=slice(4, 12))
    return ds.isel(time=slice(0, 7))

In [6]:
def plot_variable_time_step(time_index, ds, variable_name, cmap, norm):
    """
    Plot a 2D map of a selected variable from an xarray.Dataset at a specific time.

    Parameters:
    - time_index (int): Time index to plot.
    - ds (xarray.Dataset): Dataset with the variable.
    - variable_name (str): Name of the variable in the dataset.
    - cmap: colourmap object. Callable with get_cmap_bdrc() function to apply BDRC cmap.
    - norm: matplotlib Norm object. Callable with get_normalised_bdrc() function to apply BDRC normalisation.

    Behavior:
    - Extracts the 2D field at the given time.
    - Applies your custom normalization.
    - Plots the field on a simple map using Cartopy.
    """

    clear_output(wait=True)

    # Extract variable at selected time
    data = ds[variable_name].isel(time=time_index)

    # Create figure and Cartopy map
    fig, ax = plt.subplots(
        figsize=(12, 6),
        subplot_kw={"projection": ccrs.PlateCarree()}
    )

    # Plot data
    mesh = ax.pcolormesh(
        ds["lon"],
        ds["lat"],
        data,
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree()
    )

    # Add map features
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=":")
    ax.set_title(f"{variable_name} at {str(data.time.values)}")

    # Add colorbar
    plt.colorbar(mesh, ax=ax, label=variable_name)

    plt.show()

In [16]:
def update(frame, input_mesh, input_title):
    """
    Update the QuadMesh colors and title for a given frame.

    Parameters:
        frame : int
            Current frame index.
        mesh : QuadMesh
            The pcolormesh object to update.
        title : Text
            The matplotlib Text object for the title.
    """
    input_mesh.set_array(ds_dod.isel(time=frame))  # update data values
    title.set_text(f'{input_title} {str(time[frame].values)}')
    return input_mesh, title

In [8]:
def set_inset_extent(ax_inset, cross, target_ratio=4/3, margin=0.1):
    """
    Sets map extent to frame the cross-section path with balanced aspect ratio.

    Parameters:
        ax_inset: The inset axes.
        cross: xarray Dataset with 'lat' and 'lon'.
        target_ratio: Desired width/height ratio (e.g., 4/3 or 16/9).
        margin: Extra padding (fraction of width/height).
    """
    lon_min, lon_max = cross['lon'].min().item(), cross['lon'].max().item()
    lat_min, lat_max = cross['lat'].min().item(), cross['lat'].max().item()

    lon_range = lon_max - lon_min
    lat_range = lat_max - lat_min

    # Determine which dimension needs to expand to meet the target aspect ratio
    actual_ratio = lon_range / lat_range
    if actual_ratio > target_ratio:
        # Too wide, pad vertically
        desired_lat_range = lon_range / target_ratio
        lat_pad = (desired_lat_range - lat_range) / 2
        lat_min -= lat_pad
        lat_max += lat_pad
    else:
        # Too tall, pad horizontally
        desired_lon_range = lat_range * target_ratio
        lon_pad = (desired_lon_range - lon_range) / 2
        lon_min -= lon_pad
        lon_max += lon_pad

    # Add extra margin
    lon_margin = (lon_max - lon_min) * margin
    lat_margin = (lat_max - lat_min) * margin

    extent = [
        lon_min - lon_margin,
        lon_max + lon_margin,
        lat_min - lat_margin,
        lat_max + lat_margin
    ]

    ax_inset.set_extent(extent, crs=ccrs.PlateCarree())
